## Travel time matrix generator
This notebook generates the travel time matrices of each damage scenario

In [98]:
#pip install pyarrow

In [99]:
import wntr
import pandas as pd
import networkx as nx
import numpy as np

In [100]:
# load the wdn as INP file
wn = wntr.network.WaterNetworkModel('BBM-EPS.inp') 

In [101]:
# function to create length of valves and pumps as 5.0 meters
def sim_length(i,pipes,wn):
    if i in pipes:
        return wn.get_link(i).length
    else:
        return 5.0

In [102]:
# import the list of pipes of interest from the Excel file
file_rep = 'BPDRR_reparations.xlsx'
ds_sel = 'DS5'
FlagSave = 1
FlagPlot = 1

pipes_interest = pd.read_excel(
    file_rep,
    sheet_name=ds_sel,
    engine='openpyxl'
)

print(pipes_interest)

    Junction  Coefficient  Pipe ID
0      E3922      2.42800     3922
1      E3398      2.42800     3398
2      E3825      2.42800     3825
3      E4117      2.42800     4117
4      E1902      1.36575     1902
..       ...          ...      ...
103     E587      0.29025      587
104    E5893      0.29025     5893
105     E772      0.29025      772
106      E90      0.29025       90
107      E98      0.29025       98

[108 rows x 3 columns]


In [103]:
# List of pipe IDs
pipe_ids = pipes_interest["Pipe ID"].astype(str).tolist()

In [104]:
# Create a dictionary to store the start and end nodes for each pipe
pipe_nodes = {}

for pid in pipe_ids:

    link = wn.get_link(pid)

    pipe_nodes[pid] = {
        "start": link.start_node_name,
        "end": link.end_node_name
    }

In [105]:
# Graph where distances will be estimated
pipes  = wn.pipe_name_list
valves = wn.valve_name_list 
pumps  = wn.pump_name_list
links  = pipes + valves + pumps

start_node = [wn.get_link(i).start_node.name for i in pipes_interest['Pipe ID']]
final_node = [wn.get_link(i).end_node.name for i in pipes_interest['Pipe ID']]

l_dir = {(wn.get_link(i).start_node.name, 
          wn.get_link(i).end_node.name,
          sim_length(i,pipes,wn)) for i in links}

H = nx.Graph()
H.add_weighted_edges_from(l_dir)
H.number_of_nodes(), H.number_of_edges()

(4915, 6061)

In [106]:
# if loaded correctly then these values correspond to the number of reparations of the damage scenario on the excel file
len(start_node), len(final_node)

(108, 108)

In [107]:
# Create a distance matrix for the pipes of interest (not the nodes)
pipe_ids = list(pipe_nodes.keys())

distance_df = pd.DataFrame(
    index=pipe_ids,
    columns=pipe_ids,
    dtype=float
)

for p1 in pipe_ids:

    s1 = pipe_nodes[p1]["start"]
    e1 = pipe_nodes[p1]["end"]

    for p2 in pipe_ids:

        s2 = pipe_nodes[p2]["start"]
        e2 = pipe_nodes[p2]["end"]

        d1 = nx.shortest_path_length(H, s1, e2, weight="weight")
        d2 = nx.shortest_path_length(H, e1, s2, weight="weight")

        distance_df.loc[p1, p2] = (d1+d2)/2

In [108]:
# Convert meters to travel time in hours based on the speed of the crews
speed_kmh = 20.5
speed_mph = speed_kmh * 1000

travel_time = distance_df / speed_mph

In [109]:
interval = 0.25   # hours = 15 minutes

travel_time = np.ceil(travel_time / interval) * interval

travel_time

,3922,3398,3825,4117,1902,3019,91,1540,2050,2115,...,5280,5432,5523,5550,5845,587,5893,772,90,98
3922,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.25,0.25
3398,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25
3825,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.50,0.50,0.25,0.50,0.25,0.25,0.25
4117,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.50,0.25
1902,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.25,0.25,...,0.75,0.75,0.75,0.50,0.75,0.25,0.75,0.25,0.25,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.50,...,0.75,0.50,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25
5893,0.75,0.50,0.50,0.75,0.75,0.50,0.50,0.50,0.75,0.75,...,0.25,0.25,0.25,0.25,0.25,0.50,0.25,0.50,0.50,0.50
772,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.50,...,0.75,0.75,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25
90,0.25,0.25,0.25,0.50,0.25,0.25,0.25,0.25,0.25,0.25,...,0.50,0.50,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25


In [110]:
# Save the travel time matrix to a Parquet file
outfile = f"TravelTime_{ds_sel}.parquet"

travel_time.to_parquet(outfile, index=True)

print(f"Saved {outfile}")

Saved TravelTime_DS5.parquet
